# GEE からのデータ読み込みと可視化

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astrocamp-2026-siaPpts/astrocamp-sat-gis/blob/main/public/notebooks/tutorials/00_example_gee.ipynb)

このノートブックでは Google Earth Engine (GEE) を使って以下の操作を行う。

1. GEE の認証と初期化
2. Sentinel-2 衛星画像の読み込み（日付・範囲でフィルタ）
3. NDVI（正規化植生指数）の計算
4. geemap を使った地図上での可視化
5. 画像のプロパティ確認とエクスポート

**実行前に済ませておくこと:** [GEE アカウント登録](https://earthengine.google.com/) が完了していること。

In [ ]:
# 必要なライブラリのインストール（Colab / ローカルの Jupyter どちらでも同じ）
!pip install geemap earthengine-api


In [ ]:
import ee
import geemap

# ---- GEE の認証・初期化（Colab / ローカル両対応） ----
# 既に認証済みならブラウザは開かず、既存の認証を使います。初回のみブラウザが開きます。
try:
    from env import gee_setup
    gee_setup()
except ImportError:
    # env.py 未アップロード時のフォールバック（従来のコード）
    ee.Authenticate()
    try:
        ee.Initialize()
    except Exception:
        ee.Initialize(project='ee-yourproject')  # 自分のプロジェクトID に置き換える


In [ ]:
# 解析対象の範囲を設定する（福島県浜通り周辺）
roi = ee.Geometry.Rectangle([140.5, 37.2, 141.2, 37.8])

# Sentinel-2 コレクションを読み込む
collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterDate('2024-04-01', '2024-09-30')  # 2024年の生育期
    .filterBounds(roi)                        # 対象範囲
    .filterMetadata('CLOUDY_PIXEL_PERCENTAGE', 'less_than', 20)  # 雲20%未満
)

print(f'該当画像数: {collection.size().getInfo()}')

# 最も雲の少ない画像を1枚選ぶ
image = collection.sort('CLOUDY_PIXEL_PERCENTAGE').first()
print(f'選択画像: {image.id().getInfo()}')
print(f'撮影日時: {image.date().format("YYYY-MM-dd").getInfo()}')

In [ ]:
# NDVI = (NIR - Red) / (NIR + Red)
# Sentinel-2 L2A: B8 = NIR, B4 = Red
ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')

# NDVI の統計情報を確認する
stats = ndvi.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=roi,
    scale=100,
    bestEffort=True
)
print(f'NDVI 範囲: {stats.getInfo()}')

In [ ]:
# geemap で地図を作成する
m = geemap.Map()

# 真色画像（RGB）を追加
vis_rgb = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 2000}
m.add_layer(image, vis_rgb, 'True Color (RGB)')

# NDVI を追加
vis_ndvi = {'min': -1, 'max': 1, 'palette': ['blue', 'white', 'green']}
m.add_layer(ndvi, vis_ndvi, 'NDVI')

# 地図の中心を対象範囲に設定する
m.center_object(roi)
m

In [ ]:
# 画像のメタデータを確認する
props = image.propertyNames()
print('=== 画像プロパティ一覧 ===')
for p in props.getInfo()[:10]:  # 最初の10件のみ表示
    val = image.get(p).getInfo()
    print(f'{p}: {val}')

print('\n=== 利用可能なバンド ===')
bands = image.bandNames()
for b in bands.getInfo():
    print(b)

In [ ]:
# （ローカルで実行しても、出力先は自分の Google Drive になります）
# 画像を GeoTIFF として Google Drive にエクスポートする
# ※ 実行前に Google Drive の空き容量を確認すること

task = ee.batch.Export.image.toDrive(
    image=ndvi.clip(roi),
    description='NDVI_export',
    folder='GEE_exports',
    fileNamePrefix='ndvi_fukushima',
    region=roi,
    scale=100,
    crs='EPSG:4326',
    fileFormat='GeoTIFF'
)
task.start()
print('エクスポートタスクを開始しました。')
print('Google Drive → GEE_exports フォルダに保存されます。')